# **Установка необходимых библиотек**

In [ ]:
!pip install --break-system-packages clickhouse-connect

Для занятия:

Файлы лежат в каталоге /var/lib/clickhouse/data/

# **Подключаемся к базе данных**

In [ ]:
import clickhouse_connect
import pandas as pd
import os

# Вбейте свйо телеграм никнейм или любое, чем мы можем вас различить
database = 'test_daylers'

client = clickhouse_connect.get_client(host='clickhouse01', port=8123, username=os.getenv('CLICKHOUSE_USER'), password=os.getenv('CLICKHOUSE_PASSWORD'))

# **Создаем свое окружение**

In [ ]:
client.command(f'''
    CREATE DATABASE IF NOT EXISTS {database} ON CLUSTER '{clustef}';
''')

# **Типы данных**

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.type_data;
''')

client.command(f'''
    CREATE TABLE {database}.type_data
    (
    --------------------------
    -- основные типы данных --
    --------------------------
        i Int8,                               -- Int8-256 (со знаком)
        ui UInt8,                             -- UInt8-256 (без знака)
        fl Float32,                           -- Float32-64 (для мат.расчетов, но не для финансов)  
        dc Decimal(9, 4),                      -- Decimal32-256 (точность после запятой)
        st String,                            -- имеет произвольную длинну
        fst FixedString(5),                   -- имеет фиксированную длинну
    --------------------------------
    -- дополнитеьлные типы данных --
    --------------------------------
        UID UUID,                             -- уникальный идентификатор
        ip4 IPv4,                             -- 127.0.0.1
        ip6 IPv6,                             -- f2c6:e19b:da60:52ad:2cef:62fe:0279
    --------------------------------
    --    типы даты и времени     --
    --------------------------------
        dt Date,                              -- Date32 (различаются диапозном дат)
        dtm DateTime,                         -- Сохрняет время с точностью то секунд
        dtm64 DateTime64,                     -- Сохрняет время с точностью то наносекунд
    ------------------------------------
    --    композитные типы данных             -- позволяют хранить более сложные структуры данных
    ------------------------------------
        ar Array(UInt8),                      -- массив данных
        tu Tuple(Date, UInt16, Decimal32(2)), -- кортеж
        ns Nested(                            -- Вложенные структуры
            col1 String,
            col2 UInt64,
            col3 String
            ),
    mp Map(String, Int16),                    -- хранит в данные в виде ключ -> значение
    en Enum('bad' = 2,                        -- хранит данные определенного значения
            'udovlet' = 3, 
            'good' = 4 )      
    )
    ENGINE = Log;
''')


client.command(f'''
    INSERT INTO {database}.type_data
    (
               i, ui, fl, dc, st, fst,
        UID, ip4, ip6,
        dt, dtm, dtm64,
        ar, tu, 
        ns.col1, ns.col2, ns.col3,
        mp, en
    )
    VALUES 
    (
        -100,
        200,
        3.14,
        toDecimal32(3.14, 4),
        'Пример строки',
        'ABCDE',
        generateUUIDv4(),
        '192.168.1.1',
        '2001:db8::1',
        toDate('2025-04-30'),
        toDateTime('2025-04-30 14:30:00'),
        toDateTime64('2025-04-30 14:30:00.123456', 6),
        [10, 20, 30],
        (toDate('2025-04-30'), 150, 99.99),
        ['one'],
        [123456],
        ['value1'],
        {'key1': 10, 'key2': -20},
        'udovlet'
    );
''')

In [ ]:
client.query_df(f'''
    SELECT 
      *
    FROM {database}.type_data
''')

Обращение к композитрым типам данных

In [ ]:
client.query_df(f'''
    SELECT 
        ar[1],      -- обращение к эл-там массива
        ar.size0,   -- получение размера массива
        tu,         -- чтение кортежа
        ns.col1,    -- обращение к вложенной структуре
        mp['key1'], -- получение данных из Map
        en          -- Чтение Enum
    FROM {database}.type_data
''')

# **Функции к приведению типов данных**

In [ ]:
client.query_df('''
    select '1'::Int64
''')

In [ ]:
# Тут будет ошибка
client.query_df('''
    select NULL::Int64
''')

In [ ]:
client.query_df('''
    select CAST('1'  as Int8)
''')

In [ ]:
# Тут будет ошибка
client.query_df('''
   select CAST(NULL as Int8)
''')



to<Тип данных><исключение в случае ошибки приведения типа: OrNull, OrZero, OrDefault>

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.cast_type_data;
''')

client.command(f'''
CREATE TABLE {database}.cast_type_data (
    col String
)
ENGINE = Log;
''')

client.command(f'''
INSERT INTO {database}.cast_type_data values ('1'),('2'),('1a'),('-1')
''')

In [ ]:
client.query_df(f'''
    SELECT
        col,
        toInt64OrNull(col),
        toInt8OrZero(col),
        toInt8OrDefault(col, -100),
        toUInt8(-1), toUInt8(-1.1), toUInt8(256) -- выход за пределы преобразует в значение по модулю диапозона
    FROM {database}.cast_type_data
''')

# **Создание таблиц**

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.mt;
''')

client.command(f'''
    CREATE TABLE IF NOT EXISTS mt_pt
    (
        id UInt32,
        dt date
    )
    ENGINE = MergeTree -- обязательно нужно указывать движок
    PRIMARY KEY(id)   -- не обязательное поле по умолчанию равно ORDER BY
    ORDER BY(id, dt); -- обязательно должны быть колонки в порядке из primary key
''')

client.command(f'''
    INSERT INTO mt_pt
    SELECT 
      number,
      now()::date + number,
    from numbers(100);
''')

# **Описание полей при создании таблиц**


In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.test_fields_without_ttl;
''')


client.command(f'''
    CREATE TABLE {database}.test_fields_without_ttl
    (
          col_default UInt64 DEFAULT 42
        , col_materialized UInt64 MATERIALIZED col_default * 33 -- к данной колонке можно обратиться только по имени
        , col_alias UInt64 ALIAS col_default + 1                -- к данной колонке можно обратиться только по имени
        , col_codec String CODEC(ZSTD(10))
        , col_comment Date COMMENT 'Some comment'
    )
    ENGINE = Log;
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.test_fields_without_ttl (
        col_default,
        col_codec,
        col_comment
    )
    SELECT
        number,
        'какой-то текст ' || toString(number),
        toDate(now()) + number
    FROM numbers(60);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.test_fields_without_ttl
''')

In [ ]:


client.command(f'''
    DROP TABLE IF EXISTS {database}.test_fields_with_ttl;
''')


client.command(f'''
    CREATE TABLE {database}.test_fields_with_ttl
    (
          col_default UInt64 DEFAULT 42
        , col_materialized UInt64 MATERIALIZED col_default * 33
        , col_alias UInt64 ALIAS col_default + 1
        , col_codec String CODEC(ZSTD(10))
        , col_comment Date COMMENT 'Some comment'
        , col_ttl UInt64 DEFAULT 10  TTL col_comment + INTERVAL 5 DAY
    )
    ENGINE = MergeTree()
    ORDER BY (col_default);
''')

client.command(f'''
    INSERT INTO {database}.test_fields_with_ttl (
        col_default,
        col_codec,
        col_comment,
        col_ttl
    )
        SELECT
            number,
            'какой-то текст' ||  toString(number),
            toDate(now()) - number,
            rand(1) % 100000000
        FROM numbers(20);
''')

In [ ]:
client.query_df(f'''
    SELECT 
        col_default
      , col_materialized 
      , col_alias 
      , col_codec 
      , col_comment
      , col_ttl
    FROM {database}.test_fields_with_ttl
''')

In [ ]:
client.query_df(f'''
    DESCRIBE TABLE {database}.test_fields_with_ttl -- здесь можно увидеть комментарий к столбцу
''')


# **Атрибуты при создании колонок**

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.nl_lc_tabl;
''')

client.command(f'''
    CREATE TABLE {database}.nl_lc_tabl (
        a Nullable(UInt32),         -- разрешает вставку с пропуском значения.
        b LowCardinality(String),   -- ускоряет работу малокоординальных данных(часто повторяющихся)
        c UInt32
    ) ENGINE = MergeTree 
    ORDER BY tuple(); -- определяет порядок сток по порядку вставки данных
''')

client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (NULL,'test' ,1);
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1,'test2',NULL); -- null вставится как 0. Если тип данны строка вставляется как пустое значение
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, NULL, 3); 
''')
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, 'test2', 4);
''')

In [ ]:
client.query_df(f'''
    SELECT 
        a, 
        b, 
        c 
    FROM {database}.nl_lc_tabl
''')

In [ ]:

client.command(f'''
    SET input_format_null_as_default = 0; -- параметр отвечающий за вставку пустых значений. Выполняется совместно с командой на вставку
''')
# появится ошибка при вставке
client.command(f'''
    INSERT INTO {database}.nl_lc_tabl VALUES (1, 'sdasda', NULL)
''')


In [ ]:
client.query_df(f'''
    SELECT 
        toTypeName(a), 
        toTypeName(b), 
        toTypeName(c) 
    FROM {database}.nl_lc_tabl
''')

# **Партицирование**

## Диапозоном

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_range;
''')

client.command(f'''
    CREATE TABLE {database}.table_range
    (
        id UInt32,
        name String,
        created_at Date
    )
    ENGINE = MergeTree
    PARTITION BY
        CASE
            WHEN id < 10000 THEN 'range_1'
            WHEN id < 20000 THEN 'range_2'
            ELSE 'range_3'
        END
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.table_range
    SELECT
        number AS id,
        concat('User_', toString(number)) AS name,
        today() AS created_at
    FROM
        numbers(30000)
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_range FINAL;
''')

## Интервалом

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_interval;
''')

client.command(f'''
    CREATE TABLE {database}.table_interval
    (
        id UInt32,
        amount Float32,
        sale_date Date
    )
    ENGINE = MergeTree
    PARTITION BY toYYYYMM(sale_date)
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.table_interval
    SELECT
        number AS id,
        rand() % 1000 AS amount,
        today() + (number % 90) AS sale_date
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_interval FINAL;
''')

## хеш-функцией

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_hash;
''')

client.command(f'''
    CREATE TABLE {database}.table_hash
    (
        user_id UInt64,
        event String
    )
    ENGINE = MergeTree
    PARTITION BY cityHash64(user_id) % 10
    ORDER BY user_id;
''')

client.command(f'''
    INSERT INTO {database}.table_hash
    SELECT
        number AS user_id,
        concat('event_', toString(number))
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_hash FINAL;
''')

## комбинированое 

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.table_composiste;
''')

client.command(f'''
    CREATE TABLE {database}.table_composiste
    (
        order_id UInt64,
        customer_id UInt64,
        order_date Date
    )
    ENGINE = MergeTree
    PARTITION BY (toYYYYMM(order_date), customer_id % 10)
    ORDER BY order_id;
''')

client.command(f'''
    INSERT INTO {database}.table_composiste
    SELECT
        number AS order_id,
        number % 100 AS customer_id,
        today() + (number % 90) AS order_date
    FROM numbers(1000);
''')

client.command(f'''
    OPTIMIZE TABLE {database}.table_composiste FINAL;
''')



In [ ]:
# посмотреть какие у таблицы партиции
client.query_df(f'''
    select 
        _part,
        count() 
    FROM 
        {database}.table_composiste 
    GROUP BY _part 
''')

# **Движки таблиц**

**SummingMergeTree** - таблица с группировкой одинаковых записей по ключу сортировки и применением суммы к перечисленным полям 

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.summing_mt;
''')

client.command(f'''
    CREATE TABLE {database}.summing_mt
    (
        id UInt32,
        val UInt32,
        dt datetime,
        example UInt32  -- столбец, не входящий в ключ сортировки и параметры движка
    )
    ENGINE = SummingMergeTree(val) -- сумма будет считаться по полю val, так как оно указано в качестве параметра движка 
    ORDER BY (id)
    PARTITION BY toYYYYMM(dt); -- записи по этому ключу будут группироваться
''')

client.command(f'''
    INSERT INTO {database}.summing_mt
    SELECT 
        2, 
        (number + 1) * 10, 
        now() + number * 60 * 60 * 24,
        (number + 1) * 100 
    from numbers(30);
''')

In [ ]:
client.query_df(f'''
    select * from {database}.summing_mt
''')

In [ ]:
client.command(f'''
    OPTIMIZE TABLE {database}.summing_mt; -- ручное слияние
''')

In [ ]:
client.query_df(f'''
    select * from {database}.summing_mt
''')

In [ ]:
client.query_df(f'''
    select * from {database}.summing_mt FINAL
''')

**AggregatingMergeTree** -- это таблица, которая группирует одинаковые записи по ключу сортировки и применяет агрегатные функции к полям

#### Комбинаторы агрегатных функций

In [ ]:
client.query_df('''
    with t1 as (
    select number, 
            number * 10 as colsum,
            number % 3 as coldist
    from numbers(10)
    )
    select 
        sumIf(colsum, number % 2 == 0),
        countDistinct(coldist),
        countDistinctIf(coldist, coldist % 2 = 0)
    from t1
''')

#### Агрегаторные типы данных

Комбинаторы агрегаторных типов данных:
* SimpleState — возвращает результат агрегирующей функции типа SimpleAggregateFunction.
* State — возвращает промежуточное состояние типа AggregateFunction, используется при вставке.
* Merge — берёт множество состояний, объединяет их и возвращает результат полной агрегации данных.
* MergeState — выполняет слияние промежуточных состояний агрегации, возвращает промежуточное состояние агрегации.

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.simple;
''')

client.command(f'''
    CREATE TABLE {database}.simple (
      id UInt64, 
      val_sum SimpleAggregateFunction(sum, UInt64), -- предусмотрен для хранения простых агрегатов(хранит конечное состояние)
      val_max SimpleAggregateFunction(max, UInt32)
    ) 
    ENGINE=AggregatingMergeTree 
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.simple SELECT  1, sum(number), max(number) from numbers(100)
''')
client.command(f'''
   INSERT INTO {database}.simple SELECT  1, sum(number), max(number) from numbers(100);
''')
client.command(f'''
    INSERT INTO {database}.simple SELECT  1, sum(number), max(number) from numbers(100);
''')
client.command(f'''
    INSERT INTO {database}.simple SELECT  2, sum(number), max(number) from numbers(100);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.simple FINAL
''')

In [ ]:
# лучше делать так

client.query_df(f'''
    SELECT 
        id, 
        sum(val_sum),
        max(val_max) 
    FROM {database}.simple
    GROUP BY id
''')

In [ ]:
client.command(f'''
    OPTIMIZE TABLE {database}.simple;
''')

In [ ]:
client.query_df(f'''
    select * from {database}.simple
''')

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.aggr_func_tbl;
''')

client.command(f'''
    CREATE TABLE {database}.aggr_func_tbl
    (
        id UInt64,
        val_uniq AggregateFunction(uniq, UInt64),         -- Хранит в себе промежуточное состояние данных
        val_any AggregateFunction(anyIf, String, UInt8),
        val_quant AggregateFunction(quantiles(0.5, 0.9), UInt64)
    ) ENGINE=AggregatingMergeTree 
    ORDER BY id;
''')

client.command(f'''
    INSERT INTO {database}.aggr_func_tbl
    SELECT 
        1, 
        uniqState(toUInt64(rnd)),                 -- кол-во уникальных значений
        anyIfState(toString(rnd),rnd%2=0),
        quantilesState(0.5, 0.9)(toUInt64(rnd)) 
    FROM
        (SELECT arrayJoin(arrayMap(i -> i * 10, range(10))) as rnd);
''')

In [ ]:
# вставь эту строку в бобра иначе не выполнится
client.query_df(f'''
    SELECT * FROM {database}.aggr_func_tbl FORMAT Vertical -- промежуточные значения в бинарном виде
''')

In [ ]:
client.query_df(f'''
       SELECT uniqMerge(val_uniq), 
              quantilesMerge(0.5, 0.9)(val_quant), 
              anyIfMerge(val_any) 
       FROM {database}.aggr_func_tbl
''')
 

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.simple_aggregating_table;
''')

client.command(f'''
    CREATE TABLE {database}.simple_aggregating_table
    (
        id UInt32,
        val_max SimpleAggregateFunction(max, UInt32), 
        val_min SimpleAggregateFunction(min, UInt32),
        val_sum SimpleAggregateFunction(sum, UInt64)
    )
    ENGINE = AggregatingMergeTree
    ORDER BY (id);
''')

client.command(f'''
    INSERT INTO {database}.simple_aggregating_table 
    SELECT 1, 
    (number + 1) * 100,
    (number + 1) * 1,
    (number + 1) * 10
    from numbers(3); -- 10, 20, 30
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.simple_aggregating_table
''')
 

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.aggregating_table;
''')

client.command(f'''
    CREATE TABLE {database}.aggregating_table
    (
        id UInt32,
        val_count AggregateFunction(count, UInt64),
        val_avg   AggregateFunction(avg, Float64),
        val_groupArray AggregateFunction(groupArray, UInt64)
    )
    ENGINE = AggregatingMergeTree
    ORDER BY (id);
''')

client.command(f'''
    INSERT INTO {database}.aggregating_table
    with t1 as (
    SELECT number % 4 as id,
        (number + 1) * 1 col1,
        (number + 1) * 1 col2,
        (number + 1) * 1 col3
    from numbers(10)
    )
    select
        id, 
        countState(col1),
        avgState(toFloat64(col2)), -- Обратите внимание на toFloat64. ClickHouse не может автоматически привести 
                                    -- avgState(UInt64) → avgState(Float64), даже если кажется, 
                                    -- что avg всё равно возвращает float.
        groupArrayState(col3)
    from t1
    group by id; 
''')

In [ ]:
# напоминаю что такой результат выдаст ошибку только так FORMAT Vertical

client.query_df(f'''
    SELECT  *  FROM {database}.aggregating_table
''')

In [ ]:
client.query_df(f'''
    SELECT
        id,
        countMerge(val_count)        AS count_val,
        avgMerge(val_avg)            AS avg_val,
        groupArrayMerge(val_groupArray) AS grouped_vals
    FROM {database}.aggregating_table
    group by id
''')

# получается, что вы сохранили сначала avg(10, 20, 30), а затем avg(1, 2, 3). 
# Итоговый результат, который вы получите, будет avg(1, 2, 3, 10, 20, 30). 

 **ReplacingMergeTree** -- удаляет дублирующиеся записи с одинаковым значением ключа сортировки.

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.replacing_merge_tree;
''')

client.command(f'''
    CREATE TABLE {database}.replacing_merge_tree
    (
        id UInt32,
        dt date,
        val UInt32
    )
    ENGINE = ReplacingMergeTree(id)
    ORDER BY (id, dt);
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.replacing_merge_tree
    SELECT 1, 
        now()::date,
        (number + 1) * 400
    FROM numbers(1); 
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.replacing_merge_tree
''')

In [ ]:
client.command(f'''
    OPTIMIZE TABLE {database}.replacing_merge_tree;
''')

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.replacing_merge_tree_with_version;
''')

client.command(f'''
    CREATE TABLE {database}.replacing_merge_tree_with_version
    (
        id UInt32,
        dt date,
        val UInt32
    )
    ENGINE = ReplacingMergeTree(dt) -- dt может быть и числовой колонкой
    ORDER BY (id)
    PARTITION BY toYYYYMM(dt);
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.replacing_merge_tree_with_version
    SELECT 
        1, 
        now()::date + number - 15,
        (number + 1) * 1000
    FROM numbers(10);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.replacing_merge_tree_with_version
''')


In [ ]:
client.command(f'''
    OPTIMIZE TABLE {database}.replacing_merge_tree_with_version FINAL;
''')

**CollapsingMergeTree** -- Удаляет дубликаты по ключу сортировки в зависимости от флага

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.Books;
''')

client.command(f'''
    CREATE TABLE {database}.Books
    (
        ID UInt64,
        Page UInt8,
        Sign Int8 -- имеет только 2 значения "1" и "-1"
    )
    ENGINE = CollapsingMergeTree(Sign)
    ORDER BY ID;
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.Books values (1, 1, 1);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.Books
''')

In [ ]:
client.command(f'''
    INSERT INTO {database}.Books values (1, 1, -1),(1, 2, 1);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.Books
''')

In [ ]:

client.command(f'''
   OPTIMIZE TABLE {database}.Books;
''')

# в рамках ключа будет оставаться всегда последняя добавленая строка с "+1". 
# все строки с "-1" будут удалены 

**Log** -- для небольших таблиц. Каждая колока отдельный файл

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.el;
''')

client.command(f'''
    CREATE TABLE {database}.el
    (
        id UInt32,
        dt date
    )
    ENGINE = Log
''')

client.command(f'''
    INSERT INTO {database}.el
    select 
    number,
    now()::date + number,
    from numbers(10);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.el
''')

**File** -- позволяет считывать данные в формате файла

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.ef;
''')

client.command(f'''
    CREATE TABLE {database}.ef
    (
        id UInt32,
        dt date
    )
    ENGINE = File(CSV);
''')

client.command(f'''
    INSERT INTO {database}.ef
    SELECT 
      number,
      now()::date + number
    FROM numbers(10);
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.ef
''')

**Buffer** -- для укорения вставки в таблицы. Данные записываются в ОП далее сливаются в другую таблицу

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.eb;
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.ebt;
''')

client.command(f'''
    CREATE TABLE {database}.eb
    (
        id UInt32,
        dt date
    )
    ENGINE = Buffer({database}, -- имя БД
                    ebt,     -- имя таблицы для слива данных
                    16,      -- параллелизм (рекомендация 16)
                    30,      -- минимальное время слияния
                    60,      -- минимальное время слияния
                    5,       -- минимальное кол-во строк для слияния
                    10,      -- максимальное кол-во строк для слияния
                    10000,   -- минимальное кол-во байт для слияния
                    10000    -- максимальное кол-во байт для слияния
                    );
''')



In [ ]:
client.command(f'''
    INSERT INTO {database}.eb
    select 
        number,
        now()::date + number
    from numbers(1);
''')

In [ ]:
# будет ошибка
client.query_df(f'''
    SELECT * FROM {database}.eb
''')

In [ ]:
client.command(f'''
    CREATE TABLE {database}.ebt
        (
        id UInt32,
        dt date
    )
    ENGINE = Log;
''')

In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.ebt
''')

In [ ]:
client.command(f'''
    OPTIMIZE TABLE eb;
''')

**Memory** --данные хранятся только в оперативной памяти. При перезапуске CH данные будут утеряны

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.em;
''')

client.command(f'''
    CREATE TABLE {database}.em
    (
        id UInt32,
        dt date
    )
    ENGINE = Memory;
''')

client.command(f'''
    INSERT INTO {database}.em
    SELECT 
        number,
        now()::date + number
    FROM numbers(100);
''')





In [ ]:
client.query_df(f'''
    SELECT * FROM {database}.em
''')

**Set** -- Движок предназначен для использования в правой части оператора IN. Не хранятся дублирующие значения

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.es;
''')

client.command(f'''
    CREATE TABLE {database}.es
    (
        id UInt32
    )
    ENGINE = Set
    SETTINGS persistent = 1; -- данные будут считываться из ОП
''')

client.command(f'''
    INSERT INTO {database}.es SELECT number from numbers(30);
''')

In [ ]:
# читать данные из такой таблицы нельзя, только IN только хардкор
client.query_df(f'''
    SELECT * FROM {database}.es
''')

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.est;
''')

client.command(f'''
    CREATE TABLE {database}.est
    (
        id UInt32
    )
    ENGINE = MergeTree
    ORDER BY (id);
''')

client.command(f'''
    INSERT INTO {database}.est SELECT number from numbers(300);
''')

In [ ]:
client.query_df(f'''
    SELECT *
    FROM {database}.est
    WHERE id in {database}.es
''')

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.es2;
''')

client.command(f'''
    DROP TABLE IF EXISTS {database}.est2;
''')

client.command(f'''
    CREATE TABLE {database}.es2
    (
        id UInt32,
        id2 UInt32
    )
    ENGINE = Set
    SETTINGS persistent = 1;
''')

client.command(f'''
    CREATE TABLE {database}.est2
    (
        id UInt32,
        id2 UInt32
    )
    ENGINE = MergeTree
    ORDER BY (id);
''')

client.command(f'''
    INSERT INTO {database}.es2 SELECT number, number from numbers(30);
''')

client.command(f'''
    INSERT INTO {database}.est2 SELECT number, number from numbers(300);
''')

In [ ]:
client.query_df(f'''
    SELECT *
    FROM {database}.est2
    WHERE (id, id2) in {database}.es2
''')

**GenerateRandom** -- предназначен для генерации данных в СН для дальнейших тестов

In [ ]:
client.command(f'''
    DROP TABLE IF EXISTS {database}.eg;
''')

client.command(f'''
    CREATE TABLE {database}.eg
    (
        id UInt32, 
        val String,
        dt date,
        a Float32,
        b UUID,
        c Bool,
        d IPv6,
        e IPv4,
        g Array(UInt32)
    )
    ENGINE = GenerateRandom;
''')




In [ ]:
client.query_df(f'''
    select * 
    from {database}.eg
    limit 10
''')

**PostgreSQL** -- для работы с таблицами БД PSQL

In [ ]:
import psycopg2 as ps
import pandas as pd
import os

schema = '{database}' # В расках схемы задайте свою фамилию

conn = ps.connect(host="ru.tuna.am", 
                  port = 35663, 
                database="dev", 
                user='bootcamp', 
                password='bootcamp')

cursor = conn.cursor()

cursor.execute(f'''
    CREATE SCHEMA IF NOT EXISTS {schema};
    ''')
    
cursor.execute(f'''
    DROP TABLE IF EXISTS {schema}.departments CASCADE;
''')

cursor.execute(f'''
    CREATE TABLE {schema}.departments (
        dept_id SERIAL PRIMARY KEY,
        dept_name VARCHAR(50),
        location VARCHAR(50)
    )
''')

cursor.execute(f'''
    INSERT INTO {schema}.departments (dept_name, location) VALUES
    ('HR', 'Москва'),
    ('IT', 'Санкт-Петербург'),
    ('Finance', 'Москва'),
    ('DE', 'Краснодар')
''')

conn.commit()

In [ ]:
cursor.execute(f'''
    DROP TABLE {database}.postgresql_table;
''')

cursor.execute(f'''
    CREATE TABLE {database}.postgresql_table
    (
        dept_id Int32,
        dept_name String,
        location String
    )
    ENGINE = PostgreSQL('ru.tuna.am:35663', '{database}', 'departments',  'bootcamp', 'bootcamp');
''')

In [ ]:
client.query_df(f'''
     select * from {database}.postgresql_table
''')

**Kafka**(не работает)

In [ ]:
   
drop table {database}.kafka_order_events_raw ;
CREATE TABLE {database}.kafka_order_events_raw
(
    id UInt32,
    ts UInt64
)
ENGINE = Kafka
SETTINGS
    kafka_broker_list = 'kafka:29093',
    kafka_topic_list = 'source.public.dbz_heartbeat',
    kafka_group_name = 'clickhouse_consumer_group',
    kafka_format = 'JSONEachRow';

SET stream_like_engine_allow_direct_select = 1;
SELECT * FROM {database}.kafka_order_events_raw;

create table {database}.test 
(
id UInt32,
ts UInt64
)
Engine = Log;

CREATE MATERIALIZED VIEW {database}.consumer TO {database}.test 
   AS SELECT after.id as id, after.ts as ts
   FROM {database}.kafka_order_events_raw;

select * from {database}.test

**Распределяем таблицу по шардам**

In [ ]:
#query_df -- для вывода на экран

client.command(f'''
    CREATE TABLE {database}.events ON CLUSTER 'company_cluster' (
        time DateTime,
        uid  Int64,
        type LowCardinality(String)
    )
    ENGINE = ReplicatedMergeTree('/clickhouse/tables/{cluster}/{shard}/events', '{replica}')
    PARTITION BY toDate(time)
    ORDER BY (uid);
''')

client.command(f'''
    CREATE TABLE {database}.events_distr ON CLUSTER 'company_cluster' AS {database}.events
    ENGINE = Distributed('company_cluster', {database}, events, uid);
''')

client.command(f'''
    INSERT INTO {database}.events_distr VALUES
        ('2020-01-01 10:00:00', 100, 'view'),
        ('2020-01-01 10:05:00', 101, 'view'),
        ('2020-01-01 11:00:00', 100, 'contact'),
        ('2020-01-01 12:10:00', 101, 'view'),
        ('2020-01-02 08:10:00', 100, 'view'),
        ('2020-01-03 13:00:00', 103, 'view');
''')

In [ ]:
client.query_df(f'''
select * from {database}.events_distr 
''')

In [ ]:
client.query_df(f'''
    select * from {database}.events
''')

# **JOIN**

Ошибка распредленного джойна

In [ ]:
client.command(f'''
    drop table {database}.tabl_join_local_1 on CLUSTER '{cluster}';
''')

client.command(f'''
    CREATE TABLE {database}.tabl_join_local_1 on CLUSTER '{cluster}'
    (
      id1 UInt32,
      id2 UInt32 
    )
    engine = MergeTree
    order by (id1);
''')

client.command(f'''
    drop table {database}.tabl_join_1 on CLUSTER '{cluster}';
''')

client.command(f'''
    CREATE TABLE {database}.tabl_join_1 ON CLUSTER '{cluster}' AS {database}.tabl_join_local_1
    ENGINE = Distributed('company_cluster', {database}, tabl_join_local_1, id1);
''')

client.command(f'''
    INSERT INTO {database}.tabl_join_1 values (1, 10),(2, 11),(3, 12),(4, 13),(5, 14),(6, 15),(7, 16),(8, 17),(9, 18),(0, 29)
''')

In [ ]:
client.command(f'''
    drop table {database}.tabl_join_local_2 on CLUSTER '{cluster}';
''')

client.command(f'''
    CREATE TABLE {database}.tabl_join_local_2 on CLUSTER '{cluster}'
    (
      id1 UInt32,
      id2 UInt32 
    )
    engine = MergeTree
    order by (id1);
''')

client.command(f'''
    drop table {database}.tabl_join_2 on CLUSTER '{cluster}';
''')

client.command(f'''
    CREATE TABLE {database}.tabl_join_2 ON CLUSTER '{cluster}' AS {database}.tabl_join_local_2
    ENGINE = Distributed('company_cluster', {database}, tabl_join_local_2, id2);              -- изменен парамент распределения по шардам
''')

client.command(f'''
    INSERT INTO {database}.tabl_join_2 values (1, 10),(2, 11),(3, 12),(4, 13),(5, 14),(6, 15),(7, 16),(8, 17),(9, 18),(0, 29)
''')

In [ ]:
client.command('''
    SET distributed_product_mode = 'local'  -- по умолчанию deny
''')

client.query_df(f'''
    select *
    from {database}.tabl_join_1 as t1  
      JOIN {database}.tabl_join_2 as t2
        ON t1.id1 = t2.id1
''')

**ASOF JOIN** -- приближенное значение по условию неравентсва

In [ ]:
client.query_df('''
  SELECT 
      number AS k, 
      toDateTime('2020-10-10 10:30:00') + number * 100 as ts, 
      number * 10 AS a
  FROM system.numbers LIMIT 5
''')



In [ ]:
client.query_df('''
      SELECT number AS k, 
          toDateTime('2020-10-10 10:00:00') + number * 100 + 3 as ts, 
          number * 100 AS b
      FROM system.numbers
      LIMIT 5
    UNION ALL
      SELECT number AS k, 
          toDateTime('2020-10-10 11:00:00') + number * 100 + 3 as ts, 
          number * 1000 AS b
      FROM system.numbers
      LIMIT 5
    UNION ALL
      SELECT number AS k,
          toDateTime('2020-10-10 12:00:00') + number * 100 + 3 as ts,
          number * 10000 AS b
      FROM system.numbers
      LIMIT 5
''')
 

In [ ]:
client.query_df('''
    SELECT T_A.k, T_A.ts,  T_B.ts, T_A.a, T_B.b
    FROM
        (
            SELECT number AS k, 
            toDateTime('2020-10-10 10:30:00') + number * 100 as ts, 
            number * 10 AS a
            FROM system.numbers
            LIMIT 5
        ) T_A
    ASOF JOIN
        (
            SELECT number AS k, 
            toDateTime('2020-10-10 10:00:00') + number * 100 + 3 as ts, 
            number * 100 AS b
            FROM system.numbers
            LIMIT 5
            UNION ALL
            SELECT number AS k, 
            toDateTime('2020-10-10 11:00:00') + number * 100 + 3 as ts, 
            number * 1000 AS b
            FROM system.numbers
            LIMIT 5
            UNION ALL
            SELECT number AS k,
            toDateTime('2020-10-10 12:00:00') + number * 100 + 3 as ts,
            number * 10000 AS b
            FROM system.numbers
            LIMIT 5
        ) T_B ON
        T_A.k = T_B.k and
        T_A.ts < T_B.ts        
    ORDER BY T_A.k
        SETTINGS join_use_nulls = 0
''')


